# Weybees Artwork Analysis — Demo

Runs both AI pipelines end-to-end against the reference images committed under `fixtures/`.

**Before running:**
1. `cp .env.example .env` and set your `GEMINI_API_KEY`.
2. `docker compose up --build` (run in a separate terminal).
3. Wait until `http://localhost:8000/health` reports all services healthy.

All images are loaded from disk and sent to the gateway as base64. This keeps the demo reproducible even if the original auction URLs go offline.

In [ ]:
import base64
import json
from pathlib import Path

import httpx

GATEWAY = "http://localhost:8000"
FIXTURES = Path("../fixtures")

def as_data_uri(name: str) -> str:
    p = FIXTURES / name
    mime = "image/webp" if p.suffix == ".webp" else "image/jpeg"
    return f"data:{mime};base64," + base64.b64encode(p.read_bytes()).decode("ascii")

def show(label, payload):
    print(f"\n=== {label} ===")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

with httpx.Client(timeout=180.0) as client:
    print(client.get(f"{GATEWAY}/health").json())

## Task 1 — Artwork content extraction

Run on the Yvon GRAC beach scene (a stand-in for Guy DUC's *La Pythonisse* — see `fixtures/README.md` for why).

In [ ]:
with httpx.Client(timeout=180.0) as client:
    r = client.post(f"{GATEWAY}/task1/analyze", json={"image": as_data_uri("yvon_grac_beach.jpg")})
    r.raise_for_status()
    show("Task 1 — Yvon GRAC beach scene", r.json())

## Task 2 — Signature extraction across the full fixture set

Each image probes a different failure mode of the noise rejection logic:

| image | what it tests |
|---|---|
| `david_hockney_my_window.webp` | brief canary — must return `David Hockney`, exclude `my Window` and `Taschen` |
| `winter_in_the_land.jpg`       | clean baseline — single painted signature, no other text |
| `yvon_grac_beach.jpg`          | painted signature lower-left in a busy composition |
| `yvon_grac_signature_closeup.jpg` | signature fills the frame — high-confidence extraction |
| `yvon_grac_framed.jpg`         | gilded frame surrounds the artwork — must be ignored |
| `seascape_monogram.jpg`        | tiny painted monogram in a corner — lower confidence expected |

In [ ]:
fixture_names = [
    "david_hockney_my_window.webp",
    "winter_in_the_land.jpg",
    "yvon_grac_beach.jpg",
    "yvon_grac_signature_closeup.jpg",
    "yvon_grac_framed.jpg",
    "seascape_monogram.jpg",
]
payload = {"images": [as_data_uri(n) for n in fixture_names]}

with httpx.Client(timeout=600.0) as client:
    r = client.post(f"{GATEWAY}/task2/extract", json=payload)
    r.raise_for_status()
    result = r.json()

for sig in result["signatures"]:
    sig["image"] = fixture_names[sig["image_index"]]
show("Task 2 — all fixtures", result)

### Expected qualitative behaviour

- **Hockney *My Window*** → `David Hockney`. Title `my Window` and publisher `Taschen` must be excluded.
- **Winter in the Land** → `M.A. Gomez` (or full `Marco Antonio Gomez`).
- **Yvon GRAC beach** / **closeup** / **framed** → `Y. Grac` or similar across all three (the same signature). Frame, mat, and background should not pollute the output.
- **Seascape monogram** → either a low-confidence monogram entry or empty, depending on legibility.